# Visualisation Group Work

You can use this notebook as a template and add your plots in the cells below. We've already added some code to import the necessary packages and included an example plot to show you how a good plot might look. 

Now it's your turn to create your first plots with Python's plotting libraries. At the end of this exercise your notebook should contain one plot per library. Since you will share your notebook with the other groups, make sure to add comments so it's easy for them to understand your code. 

Your group number will tell you which kind of plot and dataset you should use for the exercise. 

| Group | Plot | Dataset | 
|-------|------|---------|
|  1 | Scatterplot | Seattle Weather |
|  2 | Lineplot | Seattle Weather | 
|  3 | Barchart | Seattle Weather  | 
|  4 | Geographical Maps | Airports |  


## What makes a plot good?

For this exercise the charts do not have to be particularly fancy or provide mind-blowing insights into the data, but they should contain all the elements that make a good plot.
Take the following plot as an example:

![example_plot](../assets/example_plot.png)

Like the plot above your figures should have/be...
1. ... a meaningful title.
2. ... labels (with units when necessary) on both axes. 
3. ... a legend (if necessary). Make sure it doesn't overlap other important elements.
4. ... text that is easily readable. You can change and increase the font size, rotate tick labels, flip axes etc. to improve readability. 
5. ... not overloaded with information. Try to keep it rather clean and simple.


In [ ]:
# Import necessary libraries
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

### Seattle Weather Dataset

In [ ]:
df_weather = pd.read_csv("../data_group_work/seattle-weather.csv")

df_weather.date = pd.to_datetime(df_weather.date)
df_weather["year"] = df_weather.date.dt.year
df_weather["month_year"] = df_weather["date"].dt.strftime("%Y-%m")


weather_class = {"drizzle": 1, "rain": 2, "sun": 3, "snow": 4, "fog": 5}

df_weather["weather_class"] = df_weather.weather.map(weather_class)
df_weather.head()

In [ ]:
df_weather_year = (
    df_weather.groupby(["year", "weather"])["precipitation"].sum().reset_index()
)
df_weather_year.year = df_weather_year.year.apply(lambda x: str(x))
df_weather_year

### US Airports Dataset

In [ ]:
df_airports = pd.read_csv("../data_group_work/airports.csv")
df_airports.head()

Before we start plotting we set a [colorblind friendly palette](https://towardsdatascience.com/two-simple-steps-to-create-colorblind-friendly-data-visualizations-2ed781a167ec)

In [ ]:
sns.set_theme(context="notebook", palette="magma_r")

## 1. Matplotlib

### Scatterplot

In [ ]:
from matplotlib.ticker import MaxNLocator

fig, ax = plt.subplots(figsize=(16, 10))

scatter_matplot = ax.scatter(
    df_weather.month_year,
    df_weather.temp_max,
    s=6 * df_weather.precipitation,
    alpha=0.5,
    c=df_weather.weather_class.to_list(),
    cmap="magma_r",  # we had to set the cmap by hand because the color list is not ordered
    label=df_weather.weather.to_list(),
)

plt.xticks(rotation=45, horizontalalignment="right", fontsize=13)
plt.yticks(fontsize=13)
plt.xlabel("Date", fontsize=15)
plt.ylabel("Max Temperature [°C]", fontsize=15)
plt.title("Temperature in Seattle", fontsize=20)
# Here you can set that only 40 x-ticks are shown, otherwise it will be too many
ax.xaxis.set_major_locator(MaxNLocator(40))


ax.legend(
    *scatter_matplot.legend_elements(),
    loc="upper left",
    fontsize=13,
    title="Precipitation Type",
)
plt.show();

In this scatter plot, we plot the date on the x-axis and the maximum temperature in degrees Celsius on the y-axis. Marker size encodes the amount of precipitation, and colour encodes the precipitation type.

### Lineplot

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

ax.plot(df_weather.date, df_weather.temp_max, label="Max Temperature")
ax.plot(df_weather.date, df_weather.temp_min, label="Min Temperature")
plt.xticks(rotation=45, horizontalalignment="right", fontsize=13)
plt.yticks(fontsize=13)
plt.xlabel("Date", fontsize=15)
plt.ylabel("Min and Max Temperature [°C]", fontsize=15)
plt.title("Temperature in Seattle", fontsize=20)
ax.legend(loc="upper left", fontsize=13)
plt.show();

### Barchart

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

ax.bar(
    x=df_weather_year.year, height=df_weather_year.precipitation, label="Precipitation"
)  # we did the lambda function
# to convert the year to string because otherwise it would not be a categorical variable
plt.xlabel("Date", fontsize=15)
plt.ylabel("Total precipitation", fontsize=15)
plt.title("Total Precipitation in Seattle", fontsize=20)

plt.show();

**Explanation**

Matplotlib gives the most manual control: figure size, tick rotation, colour map and legend are all set explicitly. That verbosity is the trade-off for full control, and it is why encoding a third variable (here precipitation as marker size and weather type as colour) takes several extra lines compared with the higher-level libraries below.

## 2. Seaborn

### Scatterplot

In [ ]:
# https://seaborn.pydata.org/generated/seaborn.scatterplot.html

# We can change the size of a seaborn plot with matplotlib
fig, ax = plt.subplots(figsize=(16, 10))

color_pal = sns.color_palette("colorblind", 6).as_hex()
colors = ",".join(color_pal)

scatter_sns = sns.scatterplot(
    x="month_year",
    y="temp_max",
    size="precipitation",
    sizes=(40, 400),  # here you can set the size range
    alpha=0.6,  # usefull if you have overlapping groups
    data=df_weather,
    hue="weather",
)

plt.xticks(rotation=45, horizontalalignment="right", fontsize=13)
plt.yticks(fontsize=13)

plt.xlabel("Date", fontsize=15)
plt.ylabel("Max Temperature [°C]", fontsize=15)
plt.title("Temperature in Seattle", fontsize=20)

scatter_sns.xaxis.set_major_locator(MaxNLocator(40))

plt.show();

### Lineplot

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

sns.lineplot(x="date", y="temp_max", data=df_weather, label="Max Temperature")
sns.lineplot(x="date", y="temp_min", data=df_weather, label="Min Temperature")
plt.xticks(rotation=45, horizontalalignment="right", fontsize=13)
plt.yticks(fontsize=13)
plt.xlabel("Date", fontsize=15)
plt.ylabel("Min and Max Temperature [°C]", fontsize=15)
plt.title("Temperature in Seattle", fontsize=20)
plt.legend(loc="upper left", fontsize=13)
plt.show();

### Barchart

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

sns.barplot(x="year", y="precipitation", data=df_weather_year, hue="weather")
plt.xlabel("Date", fontsize=15)
plt.ylabel("Precipitation", fontsize=15)
plt.title("Precipitation in Seattle by Year and Type", fontsize=20)
plt.legend(loc="upper left", fontsize=13)
plt.show();

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

sns.barplot(x="year", y="precipitation", data=df_weather, hue="weather")
plt.xlabel("Date", fontsize=15)
plt.ylabel("Average Precipitation", fontsize=15)
plt.title("Precipitation in Seattle by Year and Type", fontsize=20)
plt.legend(loc="upper left", fontsize=13)
plt.show();

**Explanation**

Seaborn sits on top of Matplotlib and reads directly from a DataFrame, so a single call with `size=`, `hue=` and `data=` reproduces the same multi-variable plot with far less code. You still drop down to Matplotlib to resize the figure or adjust the ticks, which shows how the two libraries complement each other.

## 3. Plotly

### Scatterplot

In [ ]:
# https://plotly.com/python/templates/
px.scatter(
    df_weather,
    x="month_year",
    y="temp_max",
    size="precipitation",
    color="weather",
    title="Temperature in Seattle",
    template="ggplot2",
    labels=dict(
        month_year="Date", temp_max="Max Temperature [°C]", weather="Precipitation Type"
    ),
)

### Lineplot

In [ ]:
line_plotly = px.line(title="Temperature in Seattle")
line_plotly.add_scatter(
    x=df_weather.date, y=df_weather.temp_max, name="Max Temperature"
)
line_plotly.add_scatter(
    x=df_weather.date, y=df_weather.temp_min, name="Min Temperature"
)
line_plotly.update_layout(xaxis_title="Date", yaxis_title="Temperature [°C]")

In [ ]:
px.line(
    df_weather_year,
    x="weather",
    y="precipitation",
    color="year",
    title="Total Precipitation in Seattle",
    template="ggplot2",
    labels=dict(year="Year", precipitation="Total Precipitation [mm]"),
)  # you can also make this plot for the wind

### Barchart

In [ ]:
px.bar(
    df_weather_year,
    x="year",
    y="precipitation",
    color="weather",
    template="ggplot2",
    labels=dict(year="Year", precipitation="Total Precipitation [mm]"),
)

### Geographical Maps

In [ ]:
df_airports.head()

In [ ]:
# Airports on a tile map. scatter_map replaces the deprecated scatter_mapbox (Plotly 6.x)
fig = px.scatter_map(
    df_airports,
    lat="latitude",
    lon="longitude",
    hover_name="name",
    hover_data=["city", "state"],
    color_discrete_sequence=["red"],
    zoom=1,
    width=800,
    height=600,
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 10, "t": 10, "l": 10, "b": 10})
fig.show()

#### US state map

In [ ]:
# US airports on a US state outline map.
# scope='usa' uses Plotly's own built-in US geometry, so no external map data is needed.
fig = px.scatter_geo(
    df_airports,
    lat="latitude",
    lon="longitude",
    scope="usa",
    hover_name="name",
    hover_data=["city", "state"],
    color_discrete_sequence=["red"],
    opacity=0.5,
    title="Number of airports in the US",
    width=800,
    height=500,
)
fig.update_layout(margin={"r": 10, "t": 40, "l": 10, "b": 10})
fig.show()

#### World map with the equal earth projection

In [ ]:
# The same airports on a world outline map using the equal earth projection.
fig = px.scatter_geo(
    df_airports,
    lat="latitude",
    lon="longitude",
    projection="equal earth",
    hover_name="name",
    hover_data=["city", "state"],
    color_discrete_sequence=["red"],
    opacity=0.5,
    title="Airport locations on a world map",
    width=800,
    height=500,
)
fig.update_geos(
    showcountries=True,
    countrycolor="black",
    showland=True,
    landcolor="lightgray",
    showocean=True,
    oceancolor="lightblue",
)
fig.update_layout(margin={"r": 10, "t": 40, "l": 10, "b": 10})
fig.show()

**Explanation**

Plotly Express is the most concise of the three and returns interactive figures (hover, zoom, pan) by default, which is why it is the natural choice for the geographical maps. `scatter_map` draws points on a tile map, while `scatter_geo` uses Plotly's built-in country and state outlines, so no external map data is needed. Compared with Matplotlib and Seaborn you trade some low-level control for interactivity and much shorter code.